# Weights and Biases

Notebook with code from [https://theaisummer.com/weights-and-biases-tutorial/](https://theaisummer.com/weights-and-biases-tutorial/)

In [2]:
# install and load wandb
# !pip install --quiet wandb
import wandb

In [ ]:
# define example net
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# Combines multiple image transformations into a pipeline.
#Here, two transformations are applied in sequence: ToTensor and Normalize
#To tensor: Converts image data from the dataset (e.g., PIL.Image) into a PyTorch tensor. Pixel values are scaled to the range [0, 1].
#transforms.Normalize((mean), (std)) --- central limit theorm to normalize
transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform) # applied transform to pipline
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

class Net(nn.Module):

    def __init__(self, fc_layer_size=84):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, fc_layer_size)
        self.fc3 = nn.Linear(fc_layer_size, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

100.0%


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [5]:
# set up device
if torch.cuda.is_available():
 dev = "cuda:0"
else:
 dev = "cpu"
print(dev)
device = torch.device(dev)

cpu


In [6]:
# init wandb
run = wandb.init(project='test')

# set config
config = wandb.config
config.learning_rate = 0.01
config.epochs = 10

# instatiate net
net = Net()
net.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(),lr=config.learning_rate)

# train
for epoch in range(config.epochs):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):

        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            wandb.log({'epoch': epoch+1, 'loss': running_loss/2000})
            wandb.watch(net, criterion, log="all")
            running_loss = 0.0

print('Finished Training')

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: li-kaizheng14131 (li-kaizheng14131-university-of-copenhagen). Use `wandb login --relogin` to force relogin


[1,  2000] loss: 2.199
[1,  4000] loss: 1.873
[1,  6000] loss: 1.668
[1,  8000] loss: 1.573
[1, 10000] loss: 1.508
[1, 12000] loss: 1.457
[2,  2000] loss: 1.396
[2,  4000] loss: 1.383
[2,  6000] loss: 1.347
[2,  8000] loss: 1.332
[2, 10000] loss: 1.322
[2, 12000] loss: 1.329
[3,  2000] loss: 1.235
[3,  4000] loss: 1.248
[3,  6000] loss: 1.217
[3,  8000] loss: 1.237
[3, 10000] loss: 1.217
[3, 12000] loss: 1.209
[4,  2000] loss: 1.141
[4,  4000] loss: 1.134
[4,  6000] loss: 1.136
[4,  8000] loss: 1.160
[4, 10000] loss: 1.121
[4, 12000] loss: 1.154
[5,  2000] loss: 1.057
[5,  4000] loss: 1.079
[5,  6000] loss: 1.065
[5,  8000] loss: 1.082
[5, 10000] loss: 1.097
[5, 12000] loss: 1.074
[6,  2000] loss: 0.983
[6,  4000] loss: 1.005
[6,  6000] loss: 1.034
[6,  8000] loss: 1.033
[6, 10000] loss: 1.046
[6, 12000] loss: 1.023
[7,  2000] loss: 0.936
[7,  4000] loss: 0.969
[7,  6000] loss: 0.970
[7,  8000] loss: 0.984
[7, 10000] loss: 0.994
[7, 12000] loss: 0.992
[8,  2000] loss: 0.882
[8,  4000] 

# Visualization

In [7]:
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

columns=['image','label']
data = []

for i, batch in enumerate(trainloader, 0):
    inputs, labels = batch[0], batch[1]
    for j, image in enumerate(inputs,0):
        data.append([wandb.Image(image),classes[labels[j].item()]])
    break

table= wandb.Table(data=data, columns=columns)
run.log({"cifar10_images": table})

# Artifacts

In [8]:
cifar10_artifact = wandb.Artifact("cifar10", type="dataset")
file_path = './data/cifar-10-batches-py'
cifar10_artifact.add_dir(file_path)
run.log_artifact(cifar10_artifact)

wandb: Adding directory to artifact (./data/cifar-10-batches-py)... Done. 0.4s


<Artifact cifar10>

# Sweeps

In [9]:
import math
sweep_config = {
                'method': 'random',
                'metric': {'goal': 'minimize', 'name': 'loss'},
                'parameters': {
                    'batch_size': {
                        'distribution': 'q_log_uniform_values',
                        'max': 256,
                        'min': 32
                    },
                    'epochs': {'value': 5},
                    'fc_layer_size': {'values': [128, 256, 512]},
                    'learning_rate': {'distribution': 'uniform',
                                      'max': 0.1,
                                      'min': 0},
                    'optimizer': {'values': ['adam', 'sgd']}
                }
 }

sweep_id = wandb.sweep(sweep_config, project="test")

Create sweep with ID: mgnb5es1
Sweep URL: https://wandb.ai/li-kaizheng14131-university-of-copenhagen/test/sweeps/mgnb5es1


In [ ]:
def train(config=None):

    with wandb.init(project='test', entity='serkar', config=config):
        config = wandb.config
        transform = transforms.Compose(
            [transforms.ToTensor(),
             transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                                download=True, transform=transform)
        trainloader = torch.utils.data.DataLoader(trainset, batch_size=config.batch_size,
                                                  shuffle=True, num_workers=2)
        testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                               download=True, transform=transform)

        net = Net(config.fc_layer_size)
        net.to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(net.parameters(), lr=config.learning_rate)

        if config.optimizer == "sgd":
            optimizer = torch.optim.SGD(net.parameters(),
                                  lr=config.learning_rate, momentum=0.9)
        elif optimizer == "adam":
            optimizer = torch.optim.Adam(net.parameters(),
                                   lr=config.learning_rate)

        wandb.watch(net, criterion, log="all")

        for epoch in range(config.epochs):  # loop over the dataset multiple times

            running_loss = 0.0

            for i, data in enumerate(trainloader, 0):

                inputs, labels = data[0].to(device), data[1].to(device)
                optimizer.zero_grad()
                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            print('[%d, %5d] loss: %.3f' %
                (epoch + 1, i + 1, running_loss / len(trainloader)))

            wandb.log({'epoch': epoch + 1, 'loss': running_loss / len(trainloader)})

        print('Finished Training')

In [11]:
# run sweep
wandb.agent(sweep_id, function=train, count=5)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
wandb: Agent Starting Run: ybpedwxm with config:
wandb: 	batch_size: 109
wandb: 	epochs: 5
wandb: 	fc_layer_size: 128
wandb: 	learning_rate: 0.08653489051911352
wandb: 	optimizer: sgd


wandb: 🚀 View run iconic-valley-3 at: https://wandb.ai/li-kaizheng14131-university-of-copenhagen/test/runs/jr2jd13t
wandb: Find logs at: wandb/run-20241126_143653-jr2jd13t/logs


wandb: WARNING Ignored wandb.init() arg project when running a sweep.
wandb: WARNING Ignored wandb.init() arg entity when running a sweep.


Files already downloaded and verified
Files already downloaded and verified
[1,   459] loss: 1.744
[2,   459] loss: 1.488
[3,   459] loss: 1.426
[4,   459] loss: 1.401
[5,   459] loss: 1.372
Finished Training


epoch,▁▃▅▆█
loss,█▃▂▂▁
epoch,5
loss,1.37167


...

wandb: Agent Starting Run: l2d6yk60 with config:
wandb: 	batch_size: 117
wandb: 	epochs: 5
wandb: 	fc_layer_size: 128
wandb: 	learning_rate: 0.09734337766559858
wandb: 	optimizer: sgd
wandb: WARNING Ignored wandb.init() arg project when running a sweep.
wandb: WARNING Ignored wandb.init() arg entity when running a sweep.


Files already downloaded and verified
Files already downloaded and verified
[1,   428] loss: 1.758
[2,   428] loss: 1.503
[3,   428] loss: 1.445
[4,   428] loss: 1.424
[5,   428] loss: 1.378
Finished Training


epoch,▁▃▅▆█
loss,█▃▂▂▁
epoch,5
loss,1.37789


Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fd8c2c453d0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fd8c2a3aa50, execution_count=11 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7fd8c2a3acc0, raw_cell="# run sweep
wandb.agent(sweep_id, function=train, .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu-22.04/mnt/c/Users/%E6%9D%8E%E6%A5%B7%E6%94%BF/Desktop/2024_Block2/DL/Jackie/WandB.ipynb#X15sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe